# 퀀트 노트북 샘플 — 마블로 코드탭

이 노트북은 마블로 코드탭의 노트북 기능을 한 번에 확인하기 위한 샘플입니다.
위에서부터 **셀을 순서대로 실행**하면 아래 기능이 전부 나타납니다.

| 확인 항목 | 어느 셀에서 |
| --- | --- |
| 마크다운 렌더 (표·목록·굵게) | 지금 이 셀 |
| `print()` 표준출력 | 1번 |
| 마지막 표현식 자동 출력 | 1번 |
| 셀 간 상태 공유 | 2번 → 3번 |
| pandas DataFrame 표 | 3번 |
| matplotlib 차트 인라인 | 4번 |
| 에러 트레이스백 | 6번 |

> 첫 실행은 파이썬 런타임과 numpy/pandas/matplotlib 를 불러오느라
> 시간이 걸립니다. 두 번째 셀부터는 빠릅니다.

네트워크는 쓰지 않습니다 — 가격 데이터는 고정 시드로 생성한 합성 데이터입니다.
**실제 시세가 아니므로 투자 판단에 쓰지 마세요.**

## 1. 런타임 확인

파이썬이 실제로 도는지부터 봅니다. 마지막 줄에 있는 표현식은
`print` 없이도 결과가 찍힙니다 (주피터와 같은 동작).

In [ ]:
import sys

print("python", sys.version.split()[0])
print("셀 실행이 되고 있습니다.")

# 마지막 표현식은 자동으로 출력된다
6 * 7

## 2. 합성 가격 데이터 만들기

기하 브라운 운동으로 250 거래일치 가격을 만듭니다.
시드를 고정했으니 몇 번을 돌려도 같은 결과가 나옵니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

DAYS = 250
START_PRICE = 10_000.0
DAILY_DRIFT = 0.0004     # 하루 평균 수익률
DAILY_VOL = 0.018        # 하루 변동성

shocks = rng.normal(DAILY_DRIFT, DAILY_VOL, DAYS)
price = START_PRICE * np.exp(np.cumsum(shocks))
dates = pd.bdate_range("2026-01-02", periods=DAYS)

df = pd.DataFrame({"price": price}, index=dates)
df["ret"] = df["price"].pct_change()
df["ma20"] = df["price"].rolling(20).mean()
df["ma60"] = df["price"].rolling(60).mean()

print(f"{len(df)}일치 생성 완료  ({dates[0]:%Y-%m-%d} ~ {dates[-1]:%Y-%m-%d})")
print(f"시작 {df['price'].iloc[0]:,.0f} → 종료 {df['price'].iloc[-1]:,.0f}")

## 3. DataFrame 표로 보기

앞 셀에서 만든 `df` 를 그대로 씁니다 — **셀 사이에 변수가 유지**됩니다.
DataFrame 을 마지막 줄에 두면 HTML 표로 렌더됩니다.

In [ ]:
summary = df[["price", "ret"]].describe().T
summary["단위"] = ["원", "비율"]

summary.round(4)

## 4. 차트 — 가격과 이동평균

`matplotlib` 으로 그린 그림은 셀 아래에 이미지로 바로 붙습니다.

In [ ]:
import matplotlib.pyplot as plt

# 차트 안 글자도 한글이 됩니다 (Pretendard 번들 — 티켓 LSq4qbuT).
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(df.index, df["price"], linewidth=1.2, label="종가")
ax.plot(df.index, df["ma20"], linewidth=1.0, label="20일 이동평균")
ax.plot(df.index, df["ma60"], linewidth=1.0, label="60일 이동평균")

ax.set_title("합성 가격과 이동평균")
ax.set_ylabel("가격 (원)")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()

# 마지막 표현식으로 figure 를 두면 인라인 이미지로 붙습니다.
# plt.show() 는 이 커널(AGG 백엔드)에서 경고만 내므로 쓰지 않습니다.
fig

## 5. 간단 백테스트 — 골든크로스

20일선이 60일선 위에 있으면 보유, 아니면 현금. 신호는 **다음 날**
적용해서 미래 정보를 쓰지 않도록 합니다.

> 합성 데이터에 대한 장난감 백테스트입니다. 수수료·슬리피지·세금이
> 전혀 없으므로 전략의 우열을 말해주지 않습니다.

In [ ]:
bt = df.dropna().copy()

# 오늘 종가로 만든 신호는 내일부터 적용 (look-ahead 방지)
bt["signal"] = (bt["ma20"] > bt["ma60"]).astype(int).shift(1).fillna(0)
bt["strategy_ret"] = bt["signal"] * bt["ret"]

bt["buy_hold"] = (1 + bt["ret"]).cumprod()
bt["strategy"] = (1 + bt["strategy_ret"]).cumprod()


def max_drawdown(curve):
    peak = curve.cummax()
    return float(((curve - peak) / peak).min())


fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bt.index, bt["buy_hold"], label="buy & hold")
ax.plot(bt.index, bt["strategy"], label="golden cross")
ax.axhline(1.0, color="gray", linewidth=0.8)
ax.set_title("Equity curve (synthetic, no costs)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig

In [ ]:
# 성과 요약 표.
# 퍼센트 서식은 .style 대신 문자열 포매팅으로 만듭니다 —
# pandas 의 .style 접근자는 jinja2 를 요구하는데 이 커널엔 없습니다.
result = pd.DataFrame(
    {
        "총수익률": [bt["buy_hold"].iloc[-1] - 1, bt["strategy"].iloc[-1] - 1],
        "최대낙폭": [max_drawdown(bt["buy_hold"]), max_drawdown(bt["strategy"])],
        "보유일수": [len(bt), int(bt["signal"].sum())],
    },
    index=["단순보유", "골든크로스"],
)

view = result.copy()
view["총수익률"] = result["총수익률"].map("{:.2%}".format)
view["최대낙폭"] = result["최대낙폭"].map("{:.2%}".format)

view

## 6. 에러는 어떻게 보이나

일부러 실패시켜 트레이스백이 읽을 만하게 나오는지 봅니다.
**이 셀은 에러가 나는 게 정상입니다.**

In [ ]:
df["존재하지_않는_컬럼"].mean()

## 7. 직접 해보기

아래 셀을 고쳐가며 실행해 보세요. 위에서 만든 `df` 와 `bt` 가 그대로 살아 있습니다.

- `DAILY_VOL` 을 키우고 2번 셀부터 다시 실행하면 낙폭이 커집니다
- 이동평균 기간(20/60)을 바꿔 5번 셀을 다시 돌려보세요
- 셀이 멈추지 않으면 커널을 재시작하면 됩니다

In [ ]:
# 월별 수익률 — 여기서부터 자유롭게 바꿔보세요
monthly = df["ret"].groupby(df.index.to_period("M")).sum()

fig, ax = plt.subplots(figsize=(9, 3.2))
colors = ["#2f9e44" if v >= 0 else "#e03131" for v in monthly]
ax.bar([str(p) for p in monthly.index], monthly.values, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Monthly return")
ax.set_ylabel("sum of daily returns")
ax.grid(axis="y", alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

fig